#4 - Limpeza na Camada Silver - Micro-Batch

---

Criamos o volume no Unity Catalog para armazenar os arquivos de checkpoint do stream, 
permitindo que o Spark retome o processamento de onde parou em caso de interrupção e processamentos futuros.

---

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.stocks.checkpoints

---

Importando todas as bibliotecas que serão utilizadas durante o notebook, em seguida definimos todas as váriaveis que serão utilizadas durante a execução.

# .
### Variáveis e suas utilizações
micro_batch_bronze_path = define caminho que será feito a leitura da tabela no Unity Catalog.

micro_batch_silver_path = define o caminho que será salvo a tabela no Unity Catalog.

checkpoint = define o caminho que será salvo os checkpoints da execução do stream.

---

In [0]:
import pyspark.sql.functions as sf
from pyspark.sql.window import Window

micro_batch_bronze_path = "workspace.stocks.micro_batch_bronze"
micro_batch_silver_path = "workspace.stocks.micro_batch_silver"
checkpoint = "/Volumes/workspace/stocks/checkpoints/micro_batch"

 ── Leitura da bronze como stream contínuo ───────────────────────────────────
 
 O readStream mantém a leitura ativa, processando novos registros conforme chegam na bronze.

 ── Cast de tipos e renomeação de colunas ────────────────────────────────────
 
 Renomeamos datetime para event_time convertendo para timestamp, adicionamos o timestamp 
 de ingestão e realizamos o cast de cada coluna para o tipo correto, 
 selecionando apenas as colunas necessárias para a camada silver.

 ── Remoção de nulos e duplicatas ────────────────────────────────────────────
 
 Removemos registros nulos e utilizamos watermark de um minuto para controle de estado 
 do stream, garantindo que cada combinação de ticker e event_time apareça somente uma vez.

 ── Enriquecimento e validação dos dados ─────────────────────────────────────
 
 Adicionamos week_year para identificar semana e ano de cada registro.
 
 Calculamos variacao_real e variacao_percent para medir a variação da ação a cada minuto.
 
 Criamos uma flag para identificar e remover registros com valores inválidos.

 ── Gravação na silver ───────────────────────────────────────────────────────
 
 Iniciamos o stream com availableNow, que processa todo o backlog disponível
 e encerra automaticamente, o checkpoint garante que a próxima execução 
 continue de onde parou sem reprocessar dados já salvos.

In [0]:
df = spark.read.table(micro_batch_bronze_path)


df = (df.withColumn("event_time", sf.to_timestamp(sf.col("datetime")))
           .withColumn("ingestao_ts", sf.current_timestamp())
           .withColumn("open", sf.col("open").cast("double"))
           .withColumn("high", sf.col("high").cast("double"))
           .withColumn("low", sf.col("low").cast("double"))
           .withColumn("close", sf.col("close").cast("double"))
           .withColumn("volume", sf.col("volume").cast("long"))
           .drop("datetime")
        ).select(
                "ticker", "event_time","open", "high", "low", "close", "volume","fonte", "ingestao_ts"
        )
print("-----Feita Cast de Tipos-----")


df = df.dropDuplicates(["ticker", "event_time"])
df = df.dropna()
print("-----Removidos Duplicatas e Nulos-----")


df = (df.withColumn("week_year", sf.concat(sf.weekofyear("event_time"), sf.lit("-"), sf.year("event_time")))
      .withColumn("variacao_real", (sf.col("close") - sf.col("open")))
      .withColumn("variacao_percent", (sf.col("variacao_real") / sf.col("open")*100))
      .withColumn("flag_valor_invalido",
                   sf.when(
                           (sf.col("open") <= 0) |
                           (sf.col("high") <= 0) |
                           (sf.col("low") <= 0) |
                           (sf.col("close") <= 0) |
                           (sf.col("volume") < 0), 
                           True
                   ).otherwise(False)
                   ))

df = df.filter(sf.col("flag_valor_invalido") == False)
df = df.drop("flag_valor_invalido")
print("-----Tabela Limpa e Enriquecida-----")


query = (df.write
        .format("delta")
        .outputMode("append") 
        .table(micro_batch_silver_path)
)

---

Leitura estática da tabela silver para
confirmar o resultado do stream.

Exibimos o total de linhas, a contagem de nulos por coluna e o Schema da tabela salva.

---

In [0]:
silver_df = spark.read.table(micro_batch_silver_path)

print(f"--Total de linhas: {silver_df.count()}")

for c in silver_df.columns:
    null_count = silver_df.filter(sf.col(c).isNull()).count()
    print(f"-Coluna '{c}': {null_count} nulos")

print("--------------------")
silver_df.printSchema()